In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import accelerate

/home/gautamk/dev/pptgenerator/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto",
    quantization_config=bnb_config,
    dtype=torch.float16
)

Loading weights: 100%|██████████| 338/338 [00:01<00:00, 259.25it/s, Materializing param=model.norm.weight]                              


In [12]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Explain transformers simply."}
]
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
def chat(model, tokenizer, message, max_length=300):
    prompt = tokenizer.apply_chat_template(
        messages,
    tokenize=False,
    add_generation_prompt = True
    )
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens = max_length,
        temperature=0.3,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id
    )
    generated_tokens = output[0][inputs["input_ids"].shape[-1]:]

    reply = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
    )
    print(reply)
    messages.append({"role": "system", "content": reply})


In [22]:
print(messages)

[{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': 'Explain transformers simply.'}, {'role': 'user', 'content': 'CAn you see the structure with which i am sending you the prompt?'}, {'role': 'system', 'content': 'Certainly! Please send your prompt so I can explain it to you in simple terms.'}, {'role': 'user', 'content': 'The prompt i am sending you right now, its made up of user and system and blah blah'}, {'role': 'system', 'content': 'Sure, I understand what you\'re trying to convey. The prompt you\'re sending is structured like this:\n\n1. **User**: This part represents the person or entity who is asking a question or providing input.\n2. **System**: This part refers to the computer program or software that processes the information from the User.\n\nSo, when you say "the prompt is made up of User and System," you\'re indicating that there\'s a conversation between these two entities. The User sends some data (like text), and then the System

In [21]:
messages.append(
    {"role": "user", "content": "Can u add more words to he body text of first slide"}
)
chat(model, tokenizer, messages)


Absolutely! Here’s a refined version of the first slide with added words and clarity:

---

### Slide 1: Introduction
- **Title:** Artificial Intelligence in 2026
- **Subtitle:** Vision and Reality
- **Body Text:**
  Welcome to our futuristic vision of AI in 2026. In this presentation, we will explore the advancements, challenges, and potential impacts of AI technology. From improved NLP to advanced robotics, quantum computing integration, ethical considerations, privacy concerns, employment impact, and global regulatory frameworks implementation, we aim to provide a balanced view of the future of AI.

---

This updated version includes more detail and clarity, making it easier to understand the objectives and expectations of the presentation. Let me know if you need any further assistance!
